In [34]:
# This snippet of code allows to output all statements in a cell
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

# SE 2026 - Lab 2 : Ranking models

We look today at Ranking Models.

In [1]:
# Top-level imports
import numpy as np
import pandas as pd
import warnings

from tqdm import tqdm

# Suppress unnecessary FutureWarning
warnings.filterwarnings('ignore', category=FutureWarning)

# Make Pandas display the entire row
pd.set_option('display.max_columns', None)

## IR Libraries

In [2]:
%%capture

# Install sdkman and use it to install Java 11
!curl -s "https://get.sdkman.io" | bash && source "$HOME/.sdkman/bin/sdkman-init.sh" && sdk install java 11.0.22-amzn < /dev/null

In [3]:
%%capture

# Install PyTerrier
!pip install python-terrier

In [4]:

%env JAVA_HOME=/root/.sdkman/candidates/java/current

# Activate PyTerrier
import pyterrier as pt
if not pt.started():
    pt.init()

env: JAVA_HOME=/root/.sdkman/candidates/java/current


/tmp/ipykernel_6108/2953124042.py:5: DeprecationWarning: Call to deprecated function (or staticmethod) started. (use pt.java.started() instead) -- Deprecated since version 0.11.0.
  if not pt.started():
Java started and loaded: pyterrier.java.colab, pyterrier.java, pyterrier.java.24, pyterrier.terrier.java [version=5.11 (build: craig.macdonald 2025-01-13 21:29), helper_version=0.0.8]
/tmp/ipykernel_6108/2953124042.py:6: DeprecationWarning: Call to deprecated method pt.init(). Deprecated since version 0.11.0.
java is now started automatically with default settings. To force initialisation early, run:
pt.java.init() # optional, forces java initialisation
  pt.init()


## The dataset

As we saw in the previous lab, we will be using a subset of the small version of [WikIR](https://www.aclweb.org/anthology/2020.lrec-1.237.pdf) dataset for English.

Download the following files (available also on Absalon under folder `lab`) in a folder called `data/`:
- `lab_docs.csv`: CSV file of document number and document text
- `lab_topics.csv`: CSV file of query id and query text
- `lab_qrels.csv`: CSV file of annotations with schema `qid, docno, label, iteration`

Let us review quickly what the dataset looks like:

In [5]:
base = 'https://raw.githubusercontent.com/Legenden84/search-engines/master/lab_test'

docs = pd.read_csv(f'{base}/lab_docs.csv', dtype={'docno': str})
topics = pd.read_csv(f'{base}/lab_topics.csv', dtype={'qid': str})
qrels = pd.read_csv(f'{base}/lab_qrels.csv')

In [6]:
print('docs shape: ', docs.shape)
docs.sample(5)
print('topics shape: ', topics.shape)
topics.sample(5)
print('qrels shape: ', qrels.shape)
qrels.sample(5)

docs shape:  (2453, 2)
topics shape:  (9, 2)
qrels shape:  (2454, 4)


,qid,docno,label,iteration
1451,14082,152560,1,0
647,14082,832661,1,0
480,14082,1196830,1,0
2224,1250390,2454262,1,0
6,1015979,229754,1,0


## Ranking Models

In order to experiment with some ranking models, we first need to index our documents:

In [7]:
indexer = pt.IterDictIndexer("./indices/sw_included", overwrite=True, blocks=True, stopwords=None)
index_ref = indexer.index(docs.to_dict(orient='records'))
index = pt.IndexFactory.of(index_ref)

In this exercise we will work our way to building our own BM25 model. Let us recall the BM25 formula:

\begin{align}
    R(q,d) \sim f_{t,q} \ \times \ \frac{f_{t,d}(k_1 + 1)}{k_1\big((1-b) + b(\frac{l_d}{l_{\text{avg}}})\big) + f_{t,d}} \ \times \ (1 + \log f_{t,d}) \ \times \ \log \frac{N}{N_t}
\end{align}

where:

* $f_{t,q}$: frequency of term $t$ in query $q$.
* $f_{t,d}$: frequency of term $t$ in document $d$.
* $l_d$: length of document $d$ in number of terms.
* $l_{\text{avg}}$: average document length in the index.
* $k_1$: tunable parameter.
* $b$: tunable parameter.
* $N$: total number of documents in the index.
* $N_t$: total number of documents containing term $t$.

`b ∈ [0, 1]` controls document-length normalization. When b = 1, full length normalization is applied — long documents are penalized heavily. When b = 0, document length is ignored entirely. Using a default value of 0.75 is a common compromise. Intuitively: b governs how suspicious we are that a high term frequency is simply due to a long document rather than genuine topical relevance.

$k1 \geq 0$ controls term-frequency saturation. Low k1 values (e.g. 0.5) cause TF to saturate quickly — a term appearing 3x is barely more important than appearing once. High k1 (e.g. 2.0) lets repeated terms keep contributing. At the extreme, k1 = 0 reduces the TF component to a binary "present or not." A default value of 1.2 is standard but can be tuned per collection.

We can think of the variables involved in the computation of BM25 as scalars ($N$, $b$, $k_1$, $l_{\text{avg}}$), vectors ($l_d$, $N_t$) and matrices ($f_{t,q}$, $f_{t,d}$).

Let us start with building the frequency term-document matrix $M_{t,d}$. Each row of this matrix should represent a term in the vocabulary and each column represents a document. Each cell represents the number of times the term $t$ appears in the document $d$, essentially $f_{t,d}$.

Using the term-document matrix $M_{t,d}$ we can easily calculate:


*   $f_{t,d}$ -  Each cell represents $f_{t,d}$
*   $l_{d}$ - Sum $M_{t,d}$ across rows. Use $l_{d}$ to get $l_{avg}$
*   $N_{t}$ - Sum $M_{t,d}$ across columns, but use binary indicator check ($f_{t,d} > 1)




In [8]:
# First, lets build a map from docid (0-based) to docno.
indexStats = index.getCollectionStatistics()
metaIndex = index.getMetaIndex()
docid2docno = {i: metaIndex.getItem("docno", i) for i in range(indexStats.getNumberOfDocuments())}

# We can fix also the scalar N
N = indexStats.getNumberOfDocuments()
print(f'N = {N}')

N = 2453


In [9]:
# indexStats gives you statistics about the index
print("====================== INDEXSTATS =========================")
print(indexStats)

====================== INDEXSTATS =========================
Number of documents: 2453
Number of terms: 23961
Number of postings: 283026
Number of fields: 0
Number of tokens: 485085
Field names: []
Positions:   true



In [10]:
# Lets print first 5 items in the docid2docno dictionary
count = 0
for docid, docno in docid2docno.items():
    print("docid: {docid} docno: {docno}".format(docid=docid, docno=docno))
    count += 1
    if count >=5:
        break

# Length of this dictionary should be equal to the number of documents
print("Number of documents: ", len(docid2docno))

docid: 0 docno: 935016
docid: 1 docno: 2360440
docid: 2 docno: 347765
docid: 3 docno: 1969335
docid: 4 docno: 1576938
Number of documents:  2453


In [11]:
# Lexicon gives you information about the vocabulary (from Lab 1)
lexicon = index.getLexicon()

# Retrieve term statistics appearing in indices (1900, 1905)
ix_range = range(1900, 1905)
for ix, kv in enumerate(index.getLexicon()):
    if ix in ix_range:
        print(f"{kv.getKey()} -> {kv.getValue().toString()}")
    elif ix > ix_range[-1]:
        break

against -> term1371 Nt=281 TF=341 maxTF=4 @{0 96998 3}
agar -> term17319 Nt=1 TF=1 maxTF=1 @{0 97818 0}
agatala -> term20566 Nt=1 TF=1 maxTF=1 @{0 97822 1}
agathonisi -> term21629 Nt=1 TF=1 maxTF=1 @{0 97826 2}
agav -> term17753 Nt=1 TF=1 maxTF=1 @{0 97830 7}


In [12]:
lexicon = index.getLexicon()

# This is our frequency term document matrix, we initialize all cells to zeros.
freq_term_doc = np.zeros((len(lexicon), N), dtype=int)

# Iterate over document IDs, retrieve its terms and update term_doc_matrix.
direct_index = index.getDirectIndex()
document_index = index.getDocumentIndex()

# Loop over every document in the index by its numeric ID (0-based)
for docid in tqdm(list(docid2docno.keys())):

    # document_index.getDocumentEntry(docid) retrieves the metadata/entry
    # for this document, which the direct index needs to locate its postings.
    #
    # direct_index.getPostings(...) then returns an iterator over all terms
    # that appear in this document. Each "posting" represents one term and
    # holds info like the term's ID and how often it occurs in this document.
    for posting in direct_index.getPostings(document_index.getDocumentEntry(docid)):

        # posting.getId() returns the numeric term ID (row in our matrix)
        termid = posting.getId()

        # posting.getFrequency() returns how many times this term appears
        # in this document, giving us the raw term frequency f_{t,d}
        freq_term_doc[termid, docid] = posting.getFrequency()

100%|██████████| 2453/2453 [00:02<00:00, 1208.24it/s]


In [13]:
print("Shape of the matrix: ", freq_term_doc.shape)
print("Maximum frequency: ", freq_term_doc.max())
print(freq_term_doc)

Shape of the matrix:  (23961, 2453)
Maximum frequency:  34
[[1 0 0 ... 0 1 0]
 [1 0 0 ... 0 0 0]
 [1 0 0 ... 0 0 0]
 ...
 [0 0 0 ... 0 0 1]
 [0 0 0 ... 0 0 1]
 [0 0 0 ... 0 0 3]]


In [14]:
# To get N_t we just sum across columns using a binary indicator
N_t = (freq_term_doc > 0).sum(axis=1)
print(f'N_t = {N_t}')
print("Shape of N_t: ", N_t.shape)

N_t = [210 234  48 ...   1   1   1]
Shape of N_t:  (23961,)


The number of non-zero values in your term-document incidence matrix should match the number of postings reported by the PyTerrier collection stats:

In [15]:
len(np.nonzero(freq_term_doc)[0]) == indexStats.numberOfPointers

True

The good thing of `freq_term_doc` is that it can help us retrieve another variable in the BM25 formula, the length of each document $l_d$.

In [16]:
l_d = freq_term_doc.sum(axis=0)
print(f'l_d = {l_d}')

l_d = [200 200 200 ... 200 200 200]


In [17]:
# Then we can get immediately l_avg
l_avg = np.mean(l_d)
print(f'l_avg = {l_avg}')

l_avg = 197.75173257236037


We are left with $f_{t,q}$ which we can also model as a matrix.

In [23]:
# Matrix of shape (num_terms x num_queries), one column per query
freq_term_query = np.zeros((len(lexicon), len(topics)), dtype=int)

invertedIndex = index.getInvertedIndex()

# Loop over each query in the topics dataframe
for i, row in pt.tqdm(enumerate(topics.itertuples(index=False)), total=len(topics)):

    # --- Query processing pipeline ---
    # PyTerrier doesn't have a simple "parse this query string" function,
    # so we manually run the same pipeline Terrier uses internally.

    # Create an empty Terrier Request object (represents a search request)
    rq = pt.terrier.J.Request()
    rq.setQueryID(row.qid)            # set the query ID (e.g., "1", "2", ...)
    rq.setOriginalQuery(row.query)     # set the raw query text (e.g., "black wall")
    rq.setIndex(index)

    # Step 1: Parse the query string into Terrier's internal query representation
    pt.terrier.J.TerrierQLParser().process(None, rq)

    # Step 2: Convert the parsed query into individual matching terms
    pt.terrier.J.TerrierQLToMatchingQueryTerms().process(None, rq)

    # Step 3: Apply the term pipeline (lowercasing, stemming, stopword removal)
    # so the query terms match how documents were indexed
    pt.terrier.J.ApplyTermPipeline().process(None, rq)

    # --- Count term frequencies in the processed query ---
    processed_terms = {}
    for term in rq.getMatchingQueryTerms():
        # Get the processed term string (e.g., "wall" after stemming)
        term_str = term.getKey().toString()
        # Look it up in the lexicon to get its numeric term ID
        termid = lexicon.getLexiconEntry(term_str).getTermId()
        # Count how many times this term appears in the query
        processed_terms[termid] = processed_terms.get(termid, 0) + 1

    # --- Fill the column for this query ---
    freq_vec = np.zeros(len(lexicon), dtype=int)
    for termid, freq in processed_terms.items():
        freq_vec[termid] = freq

    freq_term_query[:, i] = freq_vec

  0%|          | 0/9 [00:00<?, ?it/s]

In [24]:
print(freq_term_query.shape)

(23961, 9)


In [25]:
np.nonzero(freq_term_query)

(array([  26,  152,  217,  252,  401,  403,  544,  607,  607, 1025, 1073,
        1157, 1548, 1984, 1986, 2107, 5652, 5903, 9217]),
 array([0, 3, 5, 3, 5, 5, 6, 4, 8, 2, 2, 7, 0, 1, 1, 4, 2, 0, 8]))

In [26]:
freq_term_query[152]

array([0, 0, 0, 1, 0, 0, 0, 0, 0])

In [27]:
np.unique(freq_term_query)

array([0, 1])

Now we have everything we need to compute BM25 ranking for each query. Let us use the following parameters:

In [28]:
b = 0.75
k1 = 1.2

In [29]:
def BM25(freq_term_doc, freq_term_query, l_d, l_avg, N, N_t, k1, b):
    enumerator = freq_term_doc * (k1 + 1)                                               # num_terms x num_docs
    denumerator = k1 * ((1 - b) + b * l_d / l_avg) + freq_term_doc                        # num_terms x num_docs
    TF = np.multiply((enumerator / denumerator), (1 + np.log(freq_term_doc + 1e-8)))    # num_terms x num_docs
    IDF = np.log(N / N_t + 1e-10).reshape(-1, 1)                                        # num_terms x 1
    TF_IDF = np.multiply(TF, IDF)                                                       # num_terms x num_docs

    R = np.dot(
        freq_term_query.T,                                                              # num_queries x num_ters
        TF_IDF                                                                          # num_terms x num_docs
    )
    return R                                                                            # num_queries x num_docs

In [30]:
ownBM25_results = BM25(freq_term_doc, freq_term_query, l_d, l_avg, N, N_t, k1, b)
for i, docid in enumerate(np.argmax(ownBM25_results, axis=1)):
    print(f'{topics.iloc[i, 1]:20} ==> {docs.loc[docs["docno"].astype(str) == metaIndex.getItem("docno", docid), "text"].values[0]}')

president of chile   ==> the president is responsible for both the chilean government and state administration although its role and significance has changed over the history of chile as well as its position and relations with other actors in the national political organization it is one of the most prominent political figures it is also considered as one of the institutions that make up the historic constitution of chile and is essential to the country s political stability under the current constitution adopted in 1980 the president serves a four year term with immediate re election being prohibited the shorter period previously the term was six years allows for parliamentary and presidential elections to be synchronized the official seat of the president of chile is the la moneda palace in the capital santiago the constitution of 1980 and its 2005 amendment establishes the requirements for becoming president the president must be a natural born citizen of the country or else born ov

Let us see now the results from PyTerrier BM25. In PyTerrier, retrieval is done through a [Retriever](https://pyterrier.readthedocs.io/en/latest/terrier-retrieval.html#retriever) object. `pt.terrier.Retriever` takes two main arguments:
- `index_location`: can be either an `Index` or an `IndexRef` object.
- `num_results`: number of results to retrieve.
- `controls`: a dictionary with at least the key `wmodel` (the value must be a string from the list of available Terrier [weighting models](http://terrier.org/docs/current/javadoc/org/terrier/matching/models/package-summary.html))

In [31]:
BM25 = pt.terrier.Retriever(index, num_results=1, controls={'wmodel': 'BM25', 'bm25.k_1': 1.2, 'bm25.b': 0.75, 'bm25.k_3': 0})
BM25_pyterrier_results = BM25(topics)

In [35]:
with pd.option_context('display.max_colwidth', 200):
    pd.merge(BM25_pyterrier_results[['query', 'docno']], docs, on='docno', left_index=False, right_index=False)[['query', 'text']]

,query,text
0,president of chile,the president is responsible for both the chilean government and state administration although its role and significance has changed over the history of chile as well as its position and relations...
1,computer animation,the more general term computer generated imagery cgi encompasses both static scenes and dynamic images while computer animation only refers to moving images modern computer animation usually uses ...
2,2020 summer olympics,since the nation s debut in 1952 israeli athletes have appeared in every edition of the summer olympic games other than the 1980 summer olympics in moscow which it opted not to attend because of t...
3,train station,the station is located at kilometric point 20 741 of paris est mulhouse ville railway and is nearby the town of le plessis tr vise hence its name opened on the paris est mulhouse ville railway the...
4,chinese cuisine,because of the chinese diaspora and historical power of the country chinese cuisine has influenced many other cuisines in asia with modifications made to cater to local palates chinese food staple...
5,world war ii,the parts list below was published in january 1945 which supports the assumption that both types were in use at the end of world war ii identical civilian trailers dated as late as 1948 have been ...
6,painting,the medium is commonly applied to the base with a brush but other implements such as knives sponges and airbrushes can be used the final work is also called a painting painting is an important for...
7,house,they can range from simple dwellings such as rudimentary huts of nomadic tribes and the improvised shacks in shantytowns to complex fixed structures of wood brick concrete or other materials conta...
8,mexican cuisine,successive waves of other mesoamerican groups brought with them their own cooking methods these included the olmec teotihuacanos toltec huastec zapotec mixtec otomi pur pecha totonac mazatec and m...


**Why these differences?**

In [36]:
def BM25_terrier(freq_term_doc, freq_term_query, l_d, l_avg, N, N_t, k1, b):
    '''
    Implementation: https://github.com/terrier-org/terrier-core/blob/5.x/modules/core/src/main/java/org/terrier/matching/models/BM25.java#L72-L77
    '''
    enumerator = freq_term_doc * (k1 + 1)                         # num_terms x num_docs
    denumerator = k1 * (1 - b + b * l_d / l_avg) + freq_term_doc  # num_terms x num_docs
    TF = enumerator / denumerator                                 # num_terms x num_docs
    IDF = np.log2((N - N_t + 0.5) / (N_t + 0.5)).reshape(-1, 1)   # num_terms x 1
    TF_IDF = np.multiply(TF, IDF)                                 # num_terms x num_docs

    R = np.dot(
        freq_term_query.T,                                        # num_queries x num_ters
        TF_IDF                                                    # num_terms x num_docs
    )
    return R                                                      # num_queries x num_docs

In [37]:
ownBM25_terrier_results = BM25_terrier(freq_term_doc, freq_term_query, l_d, l_avg, N, N_t, k1, b)
for i, docid in enumerate(np.argmax(ownBM25_terrier_results, axis=1)):
    print(f'{topics.iloc[i, 1]:20} ==> {docs.loc[docs["docno"].astype(str) == metaIndex.getItem("docno", docid), "text"].values[0]}')

president of chile   ==> the president is responsible for both the chilean government and state administration although its role and significance has changed over the history of chile as well as its position and relations with other actors in the national political organization it is one of the most prominent political figures it is also considered as one of the institutions that make up the historic constitution of chile and is essential to the country s political stability under the current constitution adopted in 1980 the president serves a four year term with immediate re election being prohibited the shorter period previously the term was six years allows for parliamentary and presidential elections to be synchronized the official seat of the president of chile is the la moneda palace in the capital santiago the constitution of 1980 and its 2005 amendment establishes the requirements for becoming president the president must be a natural born citizen of the country or else born ov

### Significance of b and k1 in BM25

In [34]:
# Three tiny documents
documents = {
    "d1": "the cat sat on the mat",            # 6 tokens, "cat" 1x
    "d2": "the cat chased the cat around the house and the cat won",  # 12 tokens, "cat" 3x
    "d3": "the cat",                            # 2 tokens, "cat" 1x
}

query = "cat"

In [35]:
# Document stats
doc_names = list(documents.keys())
doc_lengths = np.array([len(d.split()) for d in documents.values()])  # l_d
freq_cat    = np.array([d.split().count("cat") for d in documents.values()])  # f_{t,d}

N = len(documents)
l_avg = np.mean(doc_lengths)
print(N)
print(np.round(l_avg,2))
print(freq_cat)

3
6.67
[1 3 1]


In [36]:
# BM25 TF component (ignoring IDF — same for all docs since all contain "cat")
def bm25_tf(f_td, l_d, l_avg, k1, b):
    """BM25 term-frequency component for a single term."""
    numerator   = f_td * (k1 + 1)
    denominator = k1 * ((1 - b) + b * (l_d / l_avg)) + f_td
    return numerator / denominator

In [37]:
# ============================================================
# Experiment 1: Effect of b  (fix k1 = 1.2)
# ============================================================
print("=" * 60)
print("EXPERIMENT 1: Varying b  (k1 fixed at 1.2)")
print("  b controls document-length normalization")
print("  b=0 - ignore length | b=1 - full normalization")
print("=" * 60)

k1 = 1.2
rows = []
for b in [0.0, 0.25, 0.75, 1.0]:
    scores = bm25_tf(freq_cat, doc_lengths, l_avg, k1, b)
    row = {"b": b}
    for name, score in zip(doc_names, scores):
        row[name] = round(score, 4)
    rows.append(row)



df_b = pd.DataFrame(rows).set_index("b")
df_b["ranking"] = df_b[doc_names].apply(
    lambda r: " >= ".join(r.sort_values(ascending=False).index), axis=1
)

print(df_b.to_string())
print()
print("Things to notice:")
print("b=0: d1 and d3 get the SAME score (both f=1), length is ignored.")
print("b=1: d3 (shortest doc) gets the HIGHEST score — its single")
print("mention of 'cat' is very significant relative to its length.")
print("d2 has f=3 but is also the longest doc. As b increases,")
print("its length penalty grows, offsetting the benefit of repetition.")
print()

EXPERIMENT 1: Varying b  (k1 fixed at 1.2)
  b controls document-length normalization
  b=0 - ignore length | b=1 - full normalization
          d1      d2      d3         ranking
b                                           
0.00  1.0000  1.5714  1.0000  d2 >= d1 >= d3
0.25  1.0138  1.4865  1.1055  d2 >= d3 >= d1
0.75  1.0427  1.3415  1.4013  d3 >= d2 >= d1
1.00  1.0577  1.2791  1.6176  d3 >= d2 >= d1

Things to notice:
b=0: d1 and d3 get the SAME score (both f=1), length is ignored.
b=1: d3 (shortest doc) gets the HIGHEST score — its single
mention of 'cat' is very significant relative to its length.
d2 has f=3 but is also the longest doc. As b increases,
its length penalty grows, offsetting the benefit of repetition.



In [38]:
# ============================================================
# Experiment 2: Effect of k1  (fix b = 0.75)
# ============================================================
print("=" * 60)
print("EXPERIMENT 2: Varying k1  (b fixed at 0.75)")
print("k1 controls term-frequency saturation")
print("k1=0: binary (present/absent) | large k1: raw counts matter more")
print("=" * 60)

b = 0.75
rows = []
for k1 in [0.0, 0.5, 1.2, 3.0]:
    scores = bm25_tf(freq_cat, doc_lengths, l_avg, k1, b)
    row = {"k1": k1}
    for name, score in zip(doc_names, scores):
        row[name] = round(score, 4)
    rows.append(row)

df_k1 = pd.DataFrame(rows).set_index("k1")
df_k1["ranking"] = df_k1[doc_names].apply(
    lambda r: " >= ".join(r.sort_values(ascending=False).index), axis=1
)
print(df_k1.to_string())
print()
print("Things to notice:")
print("k1=0: ALL docs score 1.0 — term frequency is completely ignored. BM25 only cares whether the term is present or not.")
print("d3 stays #1 throughout because b=0.75 rewards its short length.")
print("As k1 grows, the gap between d2 (f=3) and d1 (f=1) widens:")
print("At k1=0.5 the gap is ~0.16, at k1=3.0 it's ~0.48.")
print("Higher k1 lets repeated mentions keep contributing instead of saturating.")


EXPERIMENT 2: Varying k1  (b fixed at 0.75)
k1 controls term-frequency saturation
k1=0: binary (present/absent) | large k1: raw counts matter more
         d1      d2      d3         ranking
k1                                         
0.0  1.0000  1.0000  1.0000  d1 >= d2 >= d3
0.5  1.0256  1.1842  1.2121  d3 >= d2 >= d1
1.2  1.0427  1.3415  1.4013  d3 >= d2 >= d1
3.0  1.0596  1.5385  1.6495  d3 >= d2 >= d1

Things to notice:
k1=0: ALL docs score 1.0 — term frequency is completely ignored. BM25 only cares whether the term is present or not.
d3 stays #1 throughout because b=0.75 rewards its short length.
As k1 grows, the gap between d2 (f=3) and d1 (f=1) widens:
At k1=0.5 the gap is ~0.16, at k1=3.0 it's ~0.48.
Higher k1 lets repeated mentions keep contributing instead of saturating.


*Question: Why is d3 > d2 even for k1=3?*

### Language Models

Language models (LMs) are an alternative view to the ranking problem.

**Core idea:** Instead of matching query terms against documents using TF-IDF-style weights, we estimate a *language model* for each document — essentially a probability distribution over words. We then ask: *"How likely is it that this document's language model would generate the query?"* Documents whose language models assign higher probability to the query are ranked higher.

**The problem with raw maximum likelihood:** If we estimate the document language model purely from word counts (maximum likelihood), any query term *not* appearing in the document gets probability 0, making the entire query probability 0. This is too harsh — a document about "machine learning algorithms" shouldn't score 0 just because it never mentions the exact word "algorithms."

**Smoothing to the rescue:** To fix this, we *smooth* the document model by mixing it with the collection-wide language model. The collection language model $P(t \mid C)$ is simply the relative frequency of term $t$
across all documents combined — it tells us how common a word is in general.This way, even terms absent from a document get a small background probability. Different smoothing techniques exist (Jelinek-Mercer, Dirichlet, Absolute Discounting). We focus here on **Dirichlet smoothing**.


We will use PyTerrier's implementation of the [Dirichlet LM](http://terrier.org/docs/current/javadoc/org/terrier/matching/models/DirichletLM.html), a language model with Dirichlet smoothing, a technique that pretends that each document has an extra $\mu>0$ tokens in each document.
As a result, the impact of additional terms depends on the length of a given document: the longer the document, the lower the impact.

Dirichlet smoothing can be understood as adding $\mu$ "pseudo-counts" drawn from the collection language model to each document.

The parameter $\mu > 0$ controls **how much we trust the document vs. the collection**:

- **Small $\mu$** (e.g., 10): The document's own word frequencies dominate. Good for long documents that have enough data to "speak for themselves." But short documents may overfit to their few words.
- **Large $\mu$** (e.g., 10000): The collection model dominates. All documents look increasingly similar to the collection average. Short documents are smoothed heavily — we "don't trust" them to represent a topic on their own.
- **Moderate $\mu$** (e.g., 2000): A common default. The impact of smoothing depends on document length: short documents are smoothed more (since $\mu$ is large relative to $l_d$), long documents less.


We will see how different values of $\mu$ gives different results. It could be set as the average length of document, ~197 in our case. But, as mentioned in the [documentation](http://terrier.org/docs/v4.0/javadoc/org/terrier/matching/models/DirichletLM.html), $\mu$ varies from collection to collection. Generally it could be set as 2000.


**Ranking function**:

$$
\boxed{
\mathrm{score}(d,q)
= \sum_{t\in q}
f_{t,q}\,
\log\!\Bigl(1 + \frac{f_{t,d}}{\mu}\,\frac{l_c}{f_{t,c}}\Bigr)
\;-\;l_q\;\log\!\Bigl(1 + \frac{l_d}{\mu}\Bigr)
}
$$

* $f_{t,q}$: frequency of term $t$ in query $q$.
* $f_{t,d}$: frequency of term $t$ in document $d$.
* $μ$: Tunable parameter. Default to average document length.
* $l_{c}$: number of terms in collection.
* $f_{t,c}$: frequency of term in collection.
* $l_{q}$: number of terms in query.
* $l_{d}$: document length

### $\mu = l_{avg}$

In [37]:
DLM = pt.terrier.Retriever(index, wmodel="DirichletLM", controls={"dirichletlm.mu": l_avg})

In [38]:
# Lets retrieve top 10 documents
dlmTop10 = DLM.search("black wall").head(10)
pd.merge(dlmTop10[['query', 'docno','score']], docs, on='docno', left_index=False, right_index=False)[['query', 'text', 'score']]

,query,text,score
0,black wall,he was inspired to climb during a cycling holi...,12.055366
1,black wall,it was closed in 1892 when the community built...,11.055723
2,black wall,prior to the construction of the berlin wall i...,8.348862
3,black wall,in 2013 the rifle range which was constructed ...,7.611992
4,black wall,it was created in 1942 by members of the ak wa...,7.027148
5,black wall,in washington d c the memorial commemorates ja...,7.027148
6,black wall,it was one of a number of highly experimental ...,7.027148
7,black wall,designed in the shape of a five pointed americ...,7.027148
8,black wall,mccloy in approving the committee s recommenda...,7.027148
9,black wall,it starts in 1952 and goes until the late 1960...,6.612229


### $\mu = 2000$

In [39]:
DLM = pt.terrier.Retriever(index, wmodel="DirichletLM", controls={"dirichletlm.mu": 2000})
# Lets retrieve top 10 documents
dlmTop10 = DLM.search("black wall").head(10)
pd.merge(dlmTop10[['query', 'docno','score']], docs, on='docno', left_index=False, right_index=False)[['query', 'text', 'score']]

,query,text,score
0,black wall,he was inspired to climb during a cycling holi...,5.707629
1,black wall,prior to the construction of the berlin wall i...,4.978797
2,black wall,it was closed in 1892 when the community built...,4.804008
3,black wall,in 2013 the rifle range which was constructed ...,4.269297
4,black wall,it was created in 1942 by members of the ak wa...,3.717946
5,black wall,in washington d c the memorial commemorates ja...,3.717946
6,black wall,it was one of a number of highly experimental ...,3.717946
7,black wall,designed in the shape of a five pointed americ...,3.717946
8,black wall,mccloy in approving the committee s recommenda...,3.717946
9,black wall,it starts in 1952 and goes until the late 1960...,3.335756


*Differences: 2nd and 3rd ranks interchanged. Scores are now lower and less spread out!*

### $\mu = 3000$

In [40]:
DLM = pt.terrier.Retriever(index, wmodel="DirichletLM", controls={"dirichletlm.mu": 3000})
# Lets retrieve top 10 documents
dlmTop10 = DLM.search("black wall").head(10)
pd.merge(dlmTop10[['query', 'docno','score']], docs, on='docno', left_index=False, right_index=False)[['query', 'text', 'score']]

,query,text,score
0,black wall,he was inspired to climb during a cycling holi...,4.831821
1,black wall,prior to the construction of the berlin wall i...,4.458876
2,black wall,it was closed in 1892 when the community built...,3.969547
3,black wall,in 2013 the rifle range which was constructed ...,3.762341
4,black wall,it was created in 1942 by members of the ak wa...,3.226372
5,black wall,in washington d c the memorial commemorates ja...,3.226372
6,black wall,it was one of a number of highly experimental ...,3.226372
7,black wall,designed in the shape of a five pointed americ...,3.226372
8,black wall,mccloy in approving the committee s recommenda...,3.226372
9,black wall,it starts in 1952 and goes until the late 1960...,2.858719


*Although scores further decreased, no change in ranking on comparing with $\mu = 2000$*

### $\mu = 10000$

In [41]:
DLM = pt.terrier.Retriever(index, wmodel="DirichletLM", controls={"dirichletlm.mu": 10000})
# Lets retrieve top 10 documents
dlmTop10 = DLM.search("black wall").head(10)
pd.merge(dlmTop10[['query', 'docno','score']], docs, on='docno', left_index=False, right_index=False)[['query', 'text', 'score']]

,query,text,score
0,black wall,prior to the construction of the berlin wall i...,2.923260
1,black wall,he was inspired to climb during a cycling holi...,2.571470
2,black wall,in 2013 the rifle range which was constructed ...,2.305534
3,black wall,it was closed in 1892 when the community built...,1.917027
4,black wall,it was created in 1942 by members of the ak wa...,1.856973
5,black wall,in washington d c the memorial commemorates ja...,1.856973
6,black wall,it was one of a number of highly experimental ...,1.856973
7,black wall,designed in the shape of a five pointed americ...,1.856973
8,black wall,mccloy in approving the committee s recommenda...,1.856973
9,black wall,it starts in 1952 and goes until the late 1960...,1.566547


*1st and 2nd rankings are interchanged, 3rd and 4th rankings are interchanged when comparing with $μ = 3000$. Scores further diminished*

### $\mu = 10$

In [42]:
DLM = pt.terrier.Retriever(index, wmodel="DirichletLM", controls={"dirichletlm.mu": 10})
# Lets retrieve top 10 documents
dlmTop10 = DLM.search("black wall").head(10)
pd.merge(dlmTop10[['query', 'docno','score']], docs, on='docno', left_index=False, right_index=False)[['query', 'text', 'score']]

,query,text,score
0,black wall,he was inspired to climb during a cycling holi...,12.010090
1,black wall,it was closed in 1892 when the community built...,11.010625
2,black wall,prior to the construction of the berlin wall i...,8.325850
3,black wall,in 2013 the rifle range which was constructed ...,7.589027
4,black wall,it was created in 1942 by members of the ak wa...,7.004243
5,black wall,in washington d c the memorial commemorates ja...,7.004243
6,black wall,it was one of a number of highly experimental ...,7.004243
7,black wall,designed in the shape of a five pointed americ...,7.004243
8,black wall,mccloy in approving the committee s recommenda...,7.004243
9,black wall,it starts in 1952 and goes until the late 1960...,6.589384


*No change when comparing with $\mu = l_{avg}$*

**General Advice**: Test on different values of $\mu$. Both the $μ$ terms supporting $f_{t,q}$ and $l_{q}$ decrease when we increase $\mu$. The scores will be diminished, but the effect on ranking would be determined by the values in $f_{t,q}$ and $l_{q}$